In [2]:
import openai
from google.auth import default
import google.auth.transport.requests

def re_auth():
    project_id = "gen-lang-client-0284032230"
    location = "us-central1"

    credentials, _ = default(scopes=["https://www.googleapis.com/auth/cloud-platform"])
    credentials.refresh(google.auth.transport.requests.Request())
    
    return credentials.token, project_id, location

In [6]:
import time
import json
import os
# Assuming you have the correct library imported for your client setup (e.g., from a utility file)
# If you are using Vertex AI's managed OpenAI endpoint, the setup below is correct:
# import openai 

# --- Configuration ---
INPUT_FILE = "input" 
OUTPUT_DIR = "output" 
CHUNK_SIZE = 50 # Number of items to process in one API call
MODEL_NAME = "google/gemini-2.5-flash" # model

# --- Client Setup (Based on your provided snippet) ---

# Initial client setup
# NOTE: The 'openai' library is used here, but its initialization is directed
# towards the Vertex AI OpenAPI endpoint using custom URL.
access_token, project_id, location = re_auth()
client = openai.OpenAI(
  base_url=f"https://{location}-aiplatform.googleapis.com/v1/projects/{project_id}/locations/{location}/endpoints/openapi",
  api_key=access_token
)

# Ensure the output directory exists
os.makedirs(OUTPUT_DIR, exist_ok=True)

# 1. Load the Data from the JSON File
print(f"Loading data from {INPUT_FILE}...")
try:
    with open(INPUT_FILE, 'r') as f:
        gi_data = json.load(f)
    print(f"Successfully loaded {len(gi_data)} items.")
except FileNotFoundError:
    print(f"Error: Input file '{INPUT_FILE}' not found. Please check the path.")
    exit()
except json.JSONDecodeError:
    print(f"Error: Input file '{INPUT_FILE}' is not a valid JSON file.")
    exit()

# List to collect all enriched data
all_enriched_data = []
com = 0

# 2. Iterate and Call Gemini API
for i in range(0, len(gi_data), CHUNK_SIZE):
    # Re-authenticate every 5 chunks to ensure the token is fresh
    if com % 5 == 0 and com > 0: 
        access_token, project_id, location = re_auth()
        # Re-initialize the client with the new token
        client = openai.OpenAI(
            base_url=f"https://{location}-aiplatform.googleapis.com/v1/projects/{project_id}/locations/{location}/endpoints/openapi",
            api_key=access_token
        )
        
    end = min(i + CHUNK_SIZE, len(gi_data))
    
    # Select the current chunk of data
    data_chunk = gi_data[i:end]
    data_json_string = json.dumps(data_chunk, indent=2)
    
    print("-------------------------------")
    print(f"Processing chunk from index {i} to {end - 1}")
    
    # Define the System and User Prompts
    system_prompt = (
        "You are an expert data enricher for Indian Geographical Indication (GI) tags. "
        "Your task is to process a list of GI product objects and enrich them by adding "
        "'image_url' and a concise 'info' field, and by refining the geographical coordinates. "
        "Always respond in valid JSON format."
    )
    
    user_prompt = f'''Enrich the following list of GI tag product data.
Instructions:
- For each product, add an **"image_url"** field with a plausible, representative image URL (a placeholder is acceptable, e.g., "https://example.com/images/product_name.jpg").
- For each product, add an **"info"** field with a **short, 2-3 sentence summary** of the product, focusing on its origin, unique features, and GI tag status. The summary must be descriptive and engaging.
- **CRITICAL UPDATE:** In the **"coordinates"** array, update the **"lat"** and **"lng"** to be the latitude and longitude of the specific **city, town, or main production area** associated with the GI product, not just the state capital or a general state coordinate. For example, use coordinates for **Mysore** for 'Mysore Agarbathi' and **Darjeeling** for 'Darjeeling Tea'.

Return your response as a single JSON object with a key **"enriched_gis"** containing the updated array of objects.

Data to Enrich:
{data_json_string}
'''

    try:
        response = client.chat.completions.create(
            model=MODEL_NAME,
            messages=[
                {"role": "system", "content": system_prompt},
                {"role": "user", "content": user_prompt}
            ],
            response_format={"type": "json_object"}
        )
        
        print("got response")
        raw_content = response.choices[0].message.content
        
        # Save raw response content
        raw_file_path = os.path.join(OUTPUT_DIR, f"gi-raw-{i}-{end}.json")
        with open(raw_file_path, 'w') as f:
            f.write(raw_content)
        print(f"Saved raw response to {raw_file_path}")

        # Parse and process the response
        parsed_json = json.loads(raw_content)
        # The model is instructed to use the key "enriched_gis"
        enriched_data_chunk = parsed_json.get('enriched_gis', []) 
        
        # Append the successfully enriched data to the final list
        all_enriched_data.extend(enriched_data_chunk)
        
        print(f"Successfully processed and stored {len(enriched_data_chunk)} items.")

    except Exception as e:
        print(f"An error occurred while processing chunk {i}-{end}: {e}")
        # Optionally, save the data_chunk to a failed-to-process file for review
        
    com += 1
    time.sleep(1.5) # Wait to respect rate limits

# 3. Save the Final Complete Enriched Dataset
final_output_path = os.path.join(OUTPUT_DIR, "final_enriched_gi_data.json")
with open(final_output_path, 'w') as f:
    json.dump(all_enriched_data, f, indent=2)

print("-------------------------------")
print(f"Data enrichment complete. Total **{len(all_enriched_data)}** items processed.")
print(f"Final enriched data saved to **{final_output_path}**")

/opt/anaconda3/lib/python3.12/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


Loading data from /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/processedGIData.json...
Successfully loaded 696 items.
-------------------------------
Processing chunk from index 0 to 49
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-0-50.json
Successfully processed and stored 50 items.
-------------------------------
Processing chunk from index 50 to 99
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-50-100.json
Successfully processed and stored 50 items.
-------------------------------
Processing chunk from index 100 to 149
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-100-150.json
Successfully processed and stored 50 items.
-------------------------------
Processing chunk from index 150 to 199
got response
Saved raw response to /Users/vaibhavmandav

/opt/anaconda3/lib/python3.12/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


-------------------------------
Processing chunk from index 250 to 299
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-250-300.json
Successfully processed and stored 50 items.
-------------------------------
Processing chunk from index 300 to 349
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-300-350.json
Successfully processed and stored 50 items.
-------------------------------
Processing chunk from index 350 to 399
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-350-400.json
Successfully processed and stored 50 items.
-------------------------------
Processing chunk from index 400 to 449
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-400-450.json
Successfully processed and stored 50 items.
--------------------

/opt/anaconda3/lib/python3.12/site-packages/google/auth/_default.py:76: UserWarning: Your application has authenticated using end user credentials from Google Cloud SDK without a quota project. You might receive a "quota exceeded" or "API not enabled" error. See the following page for troubleshooting: https://cloud.google.com/docs/authentication/adc-troubleshooting/user-creds. 
  warnings.warn(_CLOUD_SDK_CREDENTIALS_WARNING)


-------------------------------
Processing chunk from index 500 to 549
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-500-550.json
Successfully processed and stored 50 items.
-------------------------------
Processing chunk from index 550 to 599
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-550-600.json
Successfully processed and stored 50 items.
-------------------------------
Processing chunk from index 600 to 649
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-600-650.json
Successfully processed and stored 50 items.
-------------------------------
Processing chunk from index 650 to 695
got response
Saved raw response to /Users/vaibhavmandavkar/VIT Classwork/personal/GI Visualizer/public/script/gi-raw-650-696.json
Successfully processed and stored 46 items.
--------------------